# Research path: RQ-A headline + liquid 10 bp footnote

**Main line (20 bp frozen cell):** RQ-A cost–CVaR Pareto (`scripts/rq_a_pareto.py`
→ `experiments/rq_a/results/`). Verdict **`dominated`**: deep policies are cheaper
on mean path cost but sit **above** the classical cost–tail hull on CVaR95.

**Footnote only (10 bp liquid):** Sepp Case I uses **k/2** in eq. (2.5); our
`hedge.rollout` ledger charges **full-rate** `cost`. Runs at `cost = 0.001` align
with Sepp liquid-index **k** numerically, not Case I economics. The brittle chain
**H → K → L → I → J** is indexed in `experiments/README.md` — do **not** promote
it to the course title (see root README).

This notebook loads committed `experiments/*/results/*.json` only.


In [ ]:
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import polars as pl

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from hedging import bench as B

print("Headline cell:", B.BenchConfig().model, "cost=", B.BenchConfig().cost)

## RQ-A (headline auxiliary)

Full-run JSON: `experiments/rq_a/results/pareto.json`. Pareto plane uses
**mean_cost** and **cvar95** from `hedge.rollout` / `hedging/metrics.py` (lower
is better on both axes).


In [ ]:
RQ_A = json.loads((ROOT / "experiments" / "rq_a" / "results" / "pareto.json").read_text())
print("verdict:", RQ_A["verdict"])
classical = pl.DataFrame(RQ_A["classical"])
deep = pl.DataFrame(RQ_A["deep"])
cb = classical.filter(pl.col("label") == "classical_best").row(0, named=True)
print("classical_best mean_cost", cb["mean_cost"], "cvar95", cb["cvar95"])
deep.select("label", "mean_cost", "cvar95")

In [ ]:
pareto_png = ROOT / "experiments" / "rq_a" / "results" / "pareto.png"
if pareto_png.is_file():
    plt.figure(figsize=(6, 4))
    plt.imshow(plt.imread(pareto_png))
    plt.axis("off")
    plt.title("RQ-A committed Pareto figure")
    plt.show()

## Liquid 10 bp footnote chain (soft)

Re-swept classical hull at `cost = 0.001`; never `classical_best.json` as baseline.
Verdict strings only — details in each folder's `NOTES.md` and `REPORT.md` spine.


In [ ]:
FOOTNOTE = [
    ("RQ-H", "rq_liquid_10bp", "liquid_10bp.json"),
    ("RQ-K", "rq_liquid_baseline", "liquid_baseline.json"),
    ("RQ-L", "rq_liquid_alpha", "liquid_alpha.json"),
    ("RQ-I", "rq_liquid_robust", "liquid_robust.json"),
    ("RQ-J", "rq_liquid_anatomy", "liquid_anatomy.json"),
]
rows = []
for tag, folder, fname in FOOTNOTE:
    path = ROOT / "experiments" / folder / "results" / fname
    payload = json.loads(path.read_text())
    rows.append({"step": tag, "folder": folder, "verdict": payload["verdict"]})
pl.DataFrame(rows)

### How to read this notebook

- **RQ-A** supports the locked title: classical cost–tail rules remain the
  reference on the 20 bp cell even when deep hedging trims mean transaction cost.
- The **liquid chain** is a sensitivity footnote; several steps fail or narrow
  quickly under robustness checks (`RQ-I`, `RQ-K`). Cite `experiments/README.md`
  and the JSON audit fields, not notebook-derived Sharpe numbers.
